In [13]:
import pandas as pd
import datetime as dt
import numpy as np
from sqlalchemy import create_engine


In [14]:
USER = "root"
PASSWORD = "Abhi8383055393"
HOST = "localhost"
PORT = "3306"
DATABASE = "Sales_and_profitability_analysis"

conn_str = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

mysql_conn = create_engine(conn_str)

In [ ]:
# orders detail table
orders = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/orders.csv")

orders['order_date'] = pd.to_datetime(orders['order_date']).dt.date

orders = orders.drop_duplicates(subset=['order_id'],keep="last")

orders.info()


# order item table
order_items = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/order_items.csv")

order_items.info()

# Product Table
products = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/products.csv")

products.info()

products[products['category'].isna() == True].count()

products.loc[products['category'].isna(),['subcategory','category']]

products[products['subcategory'] == "Storage"].head(3)

category_map = {
"Printing": "Office Supplies",
"Accessories" : "Electronics",	
"Footwear" : "Clothing",	
"Audio" : "Electronics",	
"Cleaning" : "Home Appliances",	
"Paper" : "Office Supplies",	
"Desks" : "Furniture",	
"Mobiles" : "Electronics",	
"Stationery" : "Office Supplies",	
"Storage" : "Furniture"
}

products['category'] = products['category'].fillna(products['subcategory'].map(category_map))

products.info()

products['launch_date'] = pd.to_datetime(products['launch_date']).dt.date

products.info()

# region table 
regions = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/regions.csv")
regions

# sales channel table 
sales_channels = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/sales_channels.csv")
sales_channels

# returns 
returns = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/returns.csv")
returns['return_date'] = pd.to_datetime(returns['return_date']).dt.date
returns.info()


# target 
targets = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/targets.csv")
targets.info()


# export unclean data into mysql
order_items.to_sql("order_items",mysql_conn,if_exists='replace',index=False)
products.to_sql("products",mysql_conn,if_exists='replace',index=False)
returns.to_sql("returns",mysql_conn,if_exists='replace',index=False)
targets.to_sql("targets",mysql_conn,if_exists='replace',index=False)
regions.to_sql("regions",mysql_conn,if_exists='replace',index=False)
sales_channels.to_sql("sales_channels",mysql_conn,if_exists='replace',index=False)

order_items.columns

products.columns

# merging the tables 
full_order_details = pd.merge(order_items,products,how='right',on='product_id')


main_fact_table = pd.merge(
    full_order_details,
    orders,
    how='right',
    on='order_id'
)


main_fact_table.head()

main_fact_table.isna().value_counts()

main_fact_table['order_status'] = main_fact_table['order_status'].replace("completed_","completed")

main_fact_table['order_item_id'].isna().value_counts()

null_order_item_rows = main_fact_table[main_fact_table['order_item_id'].isna() == True]
null_order_item_rows

main_fact_table.isna().value_counts()

main_fact_table['product_id'].isna().value_counts()

main_fact_table[main_fact_table['product_id'].isna() == True]

main_fact_table = main_fact_table.dropna(subset=['order_item_id'])

main_fact_table.isna().value_counts()

order_items.shape

main_fact_table.columns

main_fact_table['order_status'] = main_fact_table['order_status'].str.lower().str.replace(r"[^\w\s]", "", regex=True).str.replace(r"\s+", "_", regex=True)

main_fact_table.drop(columns=["unit_cost_x"], inplace=True)

main_fact_table.columns

# creating important metrics 
main_fact_table['gross_price'] = main_fact_table['unit_price'] * main_fact_table['quantity']
main_fact_table['discount_amt'] = main_fact_table['gross_price'] * main_fact_table['discount_pct']
main_fact_table['net_revenue'] = main_fact_table['gross_price']  -  main_fact_table['discount_amt']
main_fact_table['cogs'] = main_fact_table['unit_cost_y'] * main_fact_table['quantity']
main_fact_table['gross_profit'] = main_fact_table['net_revenue'] -  main_fact_table['cogs']
main_fact_table['profit_margin'] = main_fact_table['gross_profit'] / main_fact_table['net_revenue']

main_fact_table['order_status'].value_counts()

main_fact_table[['discount_amt', 'net_revenue', 'gross_price']].head()

# "Give me the overall sales and profitability picture."
# KPI Baseline

# Total Revenue
total_revenue_in_each_status = main_fact_table.groupby('order_status')['net_revenue'].sum().sort_values(ascending=False)

# Total Gross Profit
gross_profit_by_order_status = main_fact_table.groupby('order_status')['gross_profit'].sum().sort_values(ascending=False)

# Profit Margin %
profit_margin_by_order_status = main_fact_table.groupby('order_status')['profit_margin'].sum().sort_values(ascending=False)

# Total Units Sold
unit_solds_by_category = main_fact_table.groupby('order_status')['quantity'].sum().sort_values(ascending=False)

# Total Orders
total_orders_by_category = main_fact_table.groupby('order_status')['order_id'].count().sort_values(ascending=False)

# Total Customers
total_customers_by_category = main_fact_table.groupby('order_status')['customer_id'].count().sort_values(ascending=False)

# Average Order Value
main_fact_table['avg_order_value'] = main_fact_table['net_revenue'] / main_fact_table['order_id'].count()
avg_order_value_by_order_status = main_fact_table.groupby('order_status')['avg_order_value'].sum().sort_values(ascending=False)

# Average Discount %
avg_discount_pct_by_order_status = main_fact_table.groupby('order_status')['discount_pct'].mean().sort_values(ascending=False)

unit_solds_by_category

main_fact_table['order_month'] = pd.to_datetime(main_fact_table['order_date']).dt.strftime("%Y-%m")

# Monthly Data
main_fact_table.columns

main_fact_table['order_month'].value_counts()

monthly_summary = main_fact_table.groupby('order_month').agg(
    {
        'net_revenue':'sum',
        'gross_profit':'sum',
        'order_id':'nunique',
        'customer_id':'nunique',
        'quantity':'sum',
        'avg_order_value':'mean',
        'discount_pct':'max'
    }
).reset_index()

monthly_summary["profit_margin"] = (
    monthly_summary["gross_profit"] / monthly_summary["net_revenue"]
)
monthly_summary["avg_order_value"] = (
    monthly_summary["net_revenue"] / monthly_summary["order_id"]
)

monthly_summary = monthly_summary.sort_values("order_month").reset_index(drop=True)

monthly_summary['mom_revenue_growth_pct'] = monthly_summary['net_revenue'].pct_change() * 100
monthly_summary['mom_profit_growth_pct'] = monthly_summary['gross_profit'].pct_change() * 100
monthly_summary['margin_change_in_pct'] = monthly_summary['profit_margin'].diff()

monthly_summary.head()

# What does this suggest?
# #this compares the business growth, thier profit, and profit margin from current month to previous month to check wheather the business is going in right aspect or not. 

# Are there months where revenue increased but profit decreased?
# get the monthly based data then , check to the next 
monthly_summary[['order_month','mom_revenue_growth_pct', 'mom_profit_growth_pct', 'margin_change_in_pct']]

monthly_summary["is Rev Up and Profit down"] = np.where((monthly_summary['mom_revenue_growth_pct'] > 0)&(monthly_summary['mom_profit_growth_pct'] < 0),"Yes","No")

monthly_summary.head(10)

monthly_summary[monthly_summary['is Rev Up and Profit down'] == 'Yes']

main_fact_table.head()

# checking discount hyperthrofy
main_fact_table.columns

main_fact_table['dicount_chg'] = main_fact_table['discount_pct'].pct_change() * 100

monthly_summary.columns

monthly_summary = monthly_summary.rename(columns={'order_id':'total_orders','customer_id':'total_customers'})

discount_hypo =pd.merge(monthly_summary[['total_orders','total_customers','order_month','mom_revenue_growth_pct','mom_profit_growth_pct','margin_change_in_pct','is Rev Up and Profit down']],main_fact_table,how='inner',on='order_month')

hypo_testing = discount_hypo[ discount_hypo['is Rev Up and Profit down'] == 'Yes']

hypo_testing.head()

hypo_testing['is_dis_up_or_down'] = np.where(hypo_testing['discount_pct'] > 0, "Yes","No")

hypo_testing[hypo_testing['is_dis_up_or_down'] == 'No' ]




In [ ]:
main_fact_table.columns

In [17]:
# Task A — Grain

# orders:
# one row = it gives details about particular order, what is order date, which customer make it, which region, channel and payment method for that order and lastly is order completed or not

# order_items:
# one row = it gives the details about particular order item in orders, it have order id which tells which order is connect , product id for which product is selling, their quantity , unit price , discount % and unit cost 

# products:
# one row = give details about product , what is id, what its product name, thier category, subcategory, brand, unit cost, standard price , thier supplier and launch date 

# main_fact_table:
# one row should = it gives the all details about 1 order item , it has order_id, product name, its category, sub category , brands, its unit price, details about is order status their value , its avg order value order month discount change, etc

# Which table should be the base table for the transaction-level analysis?
#  order item should be the main transaction table, but if i need all details about its product and order status , i would consider main fact table because it contains joins between products, orders , order item table 


In [ ]:
# Task B — Join validation

# Redo only the join section.

# Before and after each merge, record:
# orders : 120000
# order_items : 256156.000000
# products : 2000.000000
# and after merginig the table
# main_fact_table : 256156.000000


In [ ]:
orders.describe()

In [ ]:
order_items.describe()

In [ ]:
products.describe()

In [ ]:
main_fact_table.describe()

In [32]:
order_items['order_item_id'].duplicated().sum()

np.int64(0)

In [ ]:
print("order_items rows:", len(order_items))
print("order_items unique IDs:", order_items['order_item_id'].nunique())

print("fact rows:", len(main_fact_table))
print("fact unique IDs:", main_fact_table['order_item_id'].nunique())

In [ ]:
main_fact_table['order_item_id'].isin(order_items['order_item_id']).value_counts()

In [41]:
main_fact_table['order_item_id'].nunique()

256125

## Fact Table Grain

The analytical fact table is built from `order_items`.

One row represents one order item.

`orders` provides order-level attributes such as:
- order date
- customer
- region
- channel
- order status

`products` provides product-level attributes such as:
- product name
- category
- subcategory
- brand

The order-item table is used as the base because the primary analysis is performed at transaction-line level.

In [46]:
# now move to KPI Design
main_fact_table['order_status'].value_counts()

order_status
completed             220727
returned               11380
partially_returned     11334
cancelled               8931
pending                 3753
Name: count, dtype: int64

In [ ]:
main_fact_table['order_status'].replace("completed_","completed").value_counts()

In [48]:
main_fact_table.columns

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'discount_pct', 'product_name', 'category', 'subcategory', 'brand',
       'unit_cost_y', 'standard_price', 'supplier_id', 'launch_date',
       'order_date', 'customer_id', 'region_id', 'channel_id',
       'payment_method', 'order_status', 'gross_price', 'discount_amt',
       'net_revenue', 'cogs', 'gross_profit', 'profit_margin',
       'avg_order_value', 'order_month', 'dicount_chg'],
      dtype='str')

In [47]:
# Business decision:
# I will include complete status because it gives the all amount which generates all sales.
# I will exclude pending, returned, partially returned, cancelled because its on transaction and does not bring the amount in the business.

In [103]:
# analytics df
analytics_df = main_fact_table[['order_item_id', 'order_id','product_id','quantity','discount_pct','discount_amt','category','order_date','customer_id','net_revenue','cogs','gross_profit','profit_margin','order_month','gross_price','order_status']]

In [104]:
analytics_df.head()

,order_item_id,order_id,product_id,quantity,discount_pct,discount_amt,category,order_date,customer_id,net_revenue,cogs,gross_profit,profit_margin,order_month,gross_price,order_status
0,OI00000001,ORD0000001,P00969,3.0,0.1,900.411,office supplies,2025-12-01,C00024,8103.699,7512.00,591.699,0.073016,2025-12,9004.11,completed
1,OI00000003,ORD0000001,P00979,1.0,0.3,450.333,Office Supplies,2025-12-01,C00024,1050.777,1117.15,-66.373,-0.063166,2025-12,1501.11,completed
2,OI00000002,ORD0000001,P01211,2.0,0.0,0.000,Home Appliances,2025-12-01,C00024,1085.340,905.84,179.500,0.165386,2025-12,1085.34,completed
3,OI00000005,ORD0000002,P00823,3.0,0.3,878.355,Furniture,2025-01-29,C06641,2049.495,1965.51,83.985,0.040978,2025-01,2927.85,completed
4,OI00000004,ORD0000002,P01465,1.0,0.2,224.800,Clothing,2025-01-29,C06641,899.200,975.46,-76.260,-0.084809,2025-01,1124.00,completed


In [105]:
# KPIs 
total_revenue = analytics_df['net_revenue'].sum() 
total_cogs = analytics_df['cogs'].sum()
gross_sales = analytics_df['gross_price'].sum()
total_profit = total_revenue - total_cogs
profit_margin = total_profit/total_revenue
total_unit_solds = analytics_df['quantity'].sum()
total_orders = analytics_df['order_id'].nunique()
total_item_orders = analytics_df['order_item_id'].nunique()
total_customers = analytics_df['customer_id'].nunique()
avg_order_value = total_revenue / total_orders
avg_discount_pct = analytics_df['discount_pct'].mean()
avg_discount_amt_per_order = analytics_df['discount_amt'].sum() / total_orders
weighted_discount_pct = (analytics_df['discount_amt'].sum() / gross_sales) * 100
avg_items_per_order = total_item_orders / total_orders
order_status_data = analytics_df.groupby('order_status', as_index=False).agg(
    orders = ('order_id','nunique'),
    customers = ('customer_id','nunique'),
    revenue = ('net_revenue','sum'),
    profit = ('gross_profit','sum')
).sort_values('revenue', ascending=False)


# KPIs in Dict
kpis  = {
    "total_revenue":total_revenue,
    "total_profit" : total_profit,
    "gross_sales" : gross_sales,
    "profit_margin_%" : profit_margin,
    "total_units_sold": total_unit_solds,
    "total_orders": total_orders,
    "total_customers": total_customers,
    "avg_order_value": avg_order_value,
    "avg_discount_pct": avg_discount_pct,
    "weighted_discount_pct": weighted_discount_pct,
    "avg_items_per_order": avg_items_per_order

}

for name, value in kpis.items():
    print(f"KPI name {name} : KPI Value is {value}")


# checking the ratios calculations 
total_revenue - total_cogs
total_revenue / total_orders

main_fact_table['order_item_id'].nunique() > total_orders

print(f"total order items in main fact table is {main_fact_table['order_item_id'].nunique()} and total orders are {total_orders}")

order_status_data

# i choose the complete orders because it have the most of the data and its also makes the amount in business, 
analytics_df.columns

analytics_df['category'].isna().value_counts()

analytics_df['order_item_id'].isin(main_fact_table['order_item_id']).value_counts()

main_fact_table.shape

# Explain why each status should or shouldn't contribute to your sales KPI.
# completed	 because it contributes the actual amount generated by company , which birngs the amount for all the expenses of company thats by i choose completed status to main status for orders 
# returned its also a good status , it gives the details about that product which is not as good as customers want, and the returned orders give us option to figureout that orders, and finally take actions to improve the less returned values
# partially_returned i would not take them, becasue it tells the orders that partially returned which is hard to classify and making difference btw them, and yes 
# cancelled	its also a good it tells what orders cant be delievered which helps to track fault products or other factors which impact on business performance 
# 3	pending i would take it if stakeholders want to track where the orders pending or how much stock we have to manage or prepare next stocks based on that


# i investigate and finds out that, main fact table also have the same rows as analytics_df, the difference of 31 is occurs when, the order talbe is not cleaned properly, now after that both data frame have equal number of rows 



KPI name total_revenue : KPI Value is 1119944505.256
KPI name total_profit : KPI Value is 209094444.21599996
KPI name gross_sales : KPI Value is 1246357013.01
KPI name profit_margin_% : KPI Value is 0.186700718861248
KPI name total_units_sold : KPI Value is 626224.0
KPI name total_orders : KPI Value is 119689
KPI name total_customers : KPI Value is 9638
KPI name avg_order_value : KPI Value is 9357.121416805221
KPI name avg_discount_pct : KPI Value is 0.10219053196681309
KPI name weighted_discount_pct : KPI Value is 10.142559991595743
KPI name avg_items_per_order : KPI Value is 2.139920961826066
total order items in main fact table is 256125 and total orders are 119689


(256125, 29)

# Monthly Performance Analysis
- "How has our sales and profitability performance changed over time?"

In [107]:
analytics_df.columns

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'discount_pct',
       'discount_amt', 'category', 'order_date', 'customer_id', 'net_revenue',
       'cogs', 'gross_profit', 'profit_margin', 'order_month', 'gross_price',
       'order_status'],
      dtype='str')

In [ ]:



monthly_data = analytics_df.groupby('order_month', as_index=False).agg(
    Revenue = ('net_revenue','sum'),
    Gross_Profit = ('gross_profit','sum'),
    Orders = ('order_id','nunique'),
    Customers = ('customer_id','nunique'),
    Unit_solds = ('quantity','sum')
)

first_month = monthly_data['order_month'].min()
latest_month = monthly_data['order_month'].max()
no_of_months = monthly_data['order_month'].nunique()
no_of_months
first_month
latest_month


31

'2024-01'

'2027-01'